Reading File

In [19]:
WFDB,CSV,MORTARA=0,0,1
n=4
file_path=f'Pats/pat{n}/pat{n}.txt'
# file_path='more/mitdb/213'
# WFDB  records
import wfdb 
if WFDB:
    record = wfdb.rdrecord(file_path)
    fs=record.fs
    print(f'fs={fs}')
    sig=record.p_signal[:,0]

# CSV records
import pandas as pd
if CSV:
    csv_df = pd.read_csv(file_path)
    if 'time' in csv_df.columns:
        csv_time = csv_df['time'].values
        csv_signals = csv_df.drop(columns=['time']).values
    else:
        csv_signals = csv_df.values
    csv_fs = 250 

# Mortara Records 
from functions import dat_reader
if MORTARA:
    fs=180
    lead='DataII'
    sig,_=dat_reader(file_path,fs,f'pat{n}',lead)
    sig=sig*-1

Preprocess

In [ ]:
from features import bessel_bandpass_filter
# Noise+baseline wander elimination (0.5-40Hz)
print('Filtering the signal...')
sig=bessel_bandpass_filter(sig,fs) 
print('Finished Filtering!')

Filtering the signal...
Finished Filtering!
Resampling...


R Detection

In [14]:
from functions import r_detection
r_peaks=r_detection(sig,fs)
print(len(r_peaks))

fs is 180 and signal length is 0.15432098765432098 hours
chunks=9 len_sig=100000
653


Beat Prediction (CNN Network)


In [18]:
import importlib
import hierarchical_qrs_classifier
importlib.reload(hierarchical_qrs_classifier)
from hierarchical_qrs_classifier import HierarchicalQRSClassifier
classifier=HierarchicalQRSClassifier()
print("Beat Annotating...")
result=classifier.predict(sig,r_peaks,fs)
r=result["abnormal_probability"]
threshold=0.000001
rest=[]
good=[]
for i in range(len(r)):
    if (r[i]>threshold):
        rest.append(r_peaks[i])
    else:
        good.append(r_peaks[i])
print("Finished Annotating!")
rest=np.array(rest)
print(f'{len(rest)} of {len(r_peaks)} detected abnormal!')

Beat Annotating...
Finished Annotating!
552 of 653 detected abnormal!


Clustering

In [ ]:
from ecg_cluster_toolkit import run_pipeline
import numpy as np
report = run_pipeline(sig, fs, r_peaks, "results/Pat5",
                      hdb_min_cluster=3, hdb_variance=0.999,
                      hdb_exc_frac=0.001, process_frac=0.10,
                      corr_thr=0.90, corr_exc_frac=0.01,
                      rms_thr=0.5)
# More Information 
valid = [r for r in report if r["kind"] == "final"]
exc   = [r for r in report if r["kind"] == "exception"]
print("valid clusters:", len(valid))
print("exception clusters:", len(exc))
print(type(valid))
print(valid)
add=valid[0]["rpeaks_npz"]
n=np.load(f'{add}',allow_pickle=True)
print(n['rpeaks'])